<a href="https://colab.research.google.com/github/hail-members/llm-based-services/blob/main/Chapter_6_gpt4all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# GPT4ALL 소개와 환경 설정

**단국대학교 — LLM기반 서비스 개발의 이해**

이 노트북에서는 다음을 실습합니다:
1. GPT4ALL Python 라이브러리 설치
2. 모델 로드 및 기본 텍스트 생성
3. 하이퍼파라미터 실험 (Temperature, Top-P, Top-K)
4. GPU(CUDA) 가속 사용

---
## 0. 환경 준비

GPT4ALL Python 라이브러리를 설치합니다. Colab 또는 로컬 환경 모두 동일합니다.

In [1]:
!pip install gpt4all

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.6 MB/s eta 0:00:00


In [2]:
!pip install "gpt4all[cuda]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 29.9 MB/s eta 0:00:00


모델 파일 저장 경로를 설정합니다. GPT4ALL GUI에서 다운로드한 모델이 있다면 해당 경로를 지정해도 됩니다.


In [2]:
import os
dirmodel = 'gpt4all_models'

---
## 1. 모델 로드 & 기본 텍스트 생성

GPT4All 객체를 생성하면서 모델 파일을 지정합니다.  
처음 실행 시 모델 파일(약 4.66GB)이 자동으로 다운로드됩니다.

```
GPT4All(모델명) → 모델 로드
model.chat_session() → 채팅 세션 시작
model.generate(프롬프트) → 텍스트 생성
```

In [3]:
from gpt4all import GPT4All
model = GPT4All("Meta-Llama-3-8B-Instruct.Q4_0.gguf") # downloads / loads a 4.66GB LLM

Downloading: 100%|██████████| 4.66G/4.66G [01:08<00:00, 67.9MiB/s]
Verifying: 100%|██████████| 4.66G/4.66G [00:13<00:00, 351MiB/s]


In [5]:
with model.chat_session():
    print(model.generate("How can I run LLMs efficiently on my laptop?", max_tokens=1024))

Large Language Models (LLMs) are powerful AI models that require significant computational resources to train and use. Running them efficiently on your laptop requires a combination of hardware, software, and technique optimizations. Here's a comprehensive guide to help you get the most out of your LLMs:

**Hardware Optimizations**

1. **Processor**: Look for laptops with at least 4-6 cores (e.g., Intel Core i7 or AMD Ryzen 7). More cores can significantly speed up computations.
2. **Memory**: Ensure your laptop has sufficient RAM (at least 16 GB, but 32 GB or more is recommended).
3. **Graphics Card**: A dedicated graphics card with at least 4-6 GB of VRAM can help accelerate computations.

**Software Optimizations**

1. **LLM Frameworks**: Use frameworks like TensorFlow, PyTorch, or JAX that are optimized for LLM training and inference.
2. **GPU Support**: Install GPU drivers (e.g., NVIDIA CUDA or AMD ROCm) to utilize your laptop's graphics card for computations.
3. **Optimized Libra

위 결과가 나오기까지 시간이 걸렸을 겁니다(CPU 기준 약 1~2분).  
이것이 **On-device LLM**입니다 — 느리지만, 내 컴퓨터에서 돌아간다!

뒤에서 GPU를 사용하면 훨씬 빨라지는 것을 확인할 수 있습니다.

---
## 2. 하이퍼파라미터 실험

슬라이드에서 배운 하이퍼파라미터들을 코드로 직접 실험해봅시다.

| 파라미터 | 역할 | 값이 낮으면 | 값이 높으면 |
|---------|------|-----------|------------|
| Temperature | 응답 다양성 조절 | 항상 같은 응답 | 다양한 응답 |
| Top P | 누적 확률 기준 토큰 선택 | 소수의 토큰만 고려 | 더 많은 토큰 고려 |
| Top K | 상위 K개 토큰만 선택 | 선택지 적음 | 선택지 많음 |

### 2-1. Temperature 실험

**temp=0.0**: 같은 질문에 항상 같은 답변이 나와야 합니다.  
5번 반복 실행하면서 결과가 동일한지 확인해보세요.

In [6]:
for i in range(5):
    with model.chat_session():
        result = model.generate("Define love", max_tokens=200, temp=0.0)
        print(f"--- 시도 {i+1} ---")
        print(result)
        print()

--- 시도 1 ---
Defining love is a challenging but fascinating task! Love is a complex and multifaceted emotion that has been debated, explored, and experienced by humans for centuries. Here's an attempt to provide a comprehensive definition:

Love is a profound emotional connection characterized by strong feelings of affection, attachment, care, commitment, and devotion towards another person, place, or thing. It encompasses various forms, including romantic love, familial love, platonic love, self-love, and even unconditional love.

Some key aspects of love include:

1. **Emotional intimacy**: A deep sense of connection and understanding with the loved one.
2. **Vulnerability**: Willingness to be open, honest, and vulnerable in front of the other person.
3. **Empathy**: The ability to understand and share the feelings, needs, and perspectives of the loved one.
4. **Acceptance**: Unconditional acceptance of the loved one's flaws, imperfections, and differences.
5. **Commitment

--- 시도 2 

**temp=1.0**: 같은 질문이지만 매번 다른 답변이 나올 수 있습니다.  
위 결과와 비교해보세요!

In [7]:
for i in range(5):
    with model.chat_session():
        result = model.generate("Define love", max_tokens=200, temp=1.0)
        print(f"--- 시도 {i+1} ---")
        print(result)
        print()

--- 시도 1 ---
The age-old question: what is love?

Love is a complex and multifaceted emotion that has been debated, explored, and experienced by humans for centuries. While it's difficult to define with absolute precision, here are some common aspects of love:

1. **Emotional connection**: Love often involves an intense emotional bond between two individuals or entities (e.g., romantic partners, family members, friends). This connection can be characterized by feelings such as warmth, tenderness, and a deep sense of understanding.
2. **Positive emotions**: Love is frequently accompanied by positive emotions like joy, happiness, contentment, and euphoria. These feelings can arise from the presence or thought of the loved one, leading to an overall sense of well-being.
3. **Unconditional acceptance**: True love often involves accepting someone for who they are, without judgment or expectation of change. This unconditional acceptance fosters a safe and supportive environment where individ

**🤔 생각해보기**: temp=0.0과 temp=1.0의 결과가 어떻게 다른가요?  
어떤 상황에서 temp를 높이거나 낮추는 것이 좋을까요?

- 고객 응대 챗봇이라면? → temp를 **낮게** (일관된 답변)
- 창작 글쓰기 도우미라면? → temp를 **높게** (다양한 표현)

### 2-2. Top P 실험

Top P는 확률이 높은 토큰부터 누적하여, 합이 p가 될 때까지의 토큰에서만 선택합니다.  
top_p=1.0이면 모든 토큰을 고려, top_p=0.1이면 상위 확률 토큰만 고려합니다.

In [8]:
# top_p = 1.0 (모든 토큰 고려)
print("=== top_p=1.0 ===")
with model.chat_session():
    print(model.generate("Define love", max_tokens=200, top_p=1.0))

=== top_p=1.0 ===
Defining love is a daunting task, as it's a complex and multifaceted emotion that can be experienced in many different ways. Here are some possible definitions of love:

1. **Strong affection or attachment**: Love is a deep emotional connection with someone or something, characterized by feelings of tenderness, warmth, and devotion.
2. **Unconditional acceptance**: Love involves accepting and embracing the other person for who they are, without judgment or expectation of change.
3. **Vulnerability and risk-taking**: Loving someone means being willing to be vulnerable and take risks in order to build a deeper connection with them.
4. **Emotional investment**: Love is an emotional investment in another person's well-being, happiness, and success.
5. **Selfless giving**: Love often involves putting the needs of others before one's own, sacrificing for their benefit, and prioritizing their welfare.

Some philosophers have offered more nuanced definitions:

* Plato believe

In [9]:
# top_p = 0.1 (상위 확률 토큰만)
print("=== top_p=0.1 ===")
with model.chat_session():
    print(model.generate("Define love", max_tokens=200, top_p=0.1))

=== top_p=0.1 ===
Defining love is a challenging but fascinating task! Love is a complex and multifaceted emotion that has been debated, explored, and experienced by humans for centuries. Here's an attempt to provide a comprehensive definition:

Love is a profound emotional connection characterized by strong feelings of affection, attachment, care, commitment, and devotion towards another person, place, or thing. It encompasses various forms, including romantic love, familial love, platonic love, self-love, and even unconditional love.

Some key aspects of love include:

1. **Emotional intimacy**: A deep sense of connection and understanding with the loved one.
2. **Vulnerability**: Willingness to be open, honest, and vulnerable in front of the other person.
3. **Empathy**: The ability to understand and share the feelings, needs, and perspectives of the loved one.
4. **Acceptance**: Unconditional acceptance of the loved one's flaws, imperfections, and differences.
5. **Commitment


### 2-3. Top K 실험

Top K는 확률이 높은 상위 K개의 토큰에서만 다음 단어를 선택합니다.  
K가 작을수록 선택지가 줄어들어 단조로운 답변이 나올 수 있습니다.

In [10]:
# top_k = 2 (상위 2개 토큰에서만 선택 — 매우 제한적)
print("=== top_k=2 ===")
with model.chat_session():
    print(model.generate("Define love", max_tokens=200, top_k=2))

=== top_k=2 ===
Defining love is a challenging but fascinating task! Love is a complex and multifaceted emotion that has been debated, explored, and experienced by humans for centuries. Here's an attempt to provide a comprehensive definition:

Love is a profound emotional connection characterized by strong feelings of affection, attachment, care, commitment, and devotion towards another person, place, or thing. It encompasses various forms, including romantic love, familial love, platonic love, self-love, and even unconditional love.

Some key aspects of love include:

1. **Emotional intimacy**: A deep sense of connection and understanding with the loved one.
2. **Vulnerability**: Willingness to be open, honest, and vulnerable in front of the other person.
3. **Empathy**: The ability to understand and share the feelings, needs, and perspectives of the loved one.
4. **Acceptance**: Unconditional acceptance of the loved one's flaws, imperfections, and differences.
5. **Commitment


In [11]:
# top_k = 40 (기본값 수준 — 충분한 선택지)
print("=== top_k=40 ===")
with model.chat_session():
    print(model.generate("Define love", max_tokens=200, top_k=40))

=== top_k=40 ===
The age-old question: what is love?

Love is a complex and multifaceted emotion that has been debated, explored, and experienced by humans for centuries. While it's difficult to define with absolute precision, here are some common aspects of love:

1. **Emotional connection**: Love often involves a strong emotional bond between two individuals, characterized by feelings of affection, tenderness, and warmth.
2. **Intimacy**: Physical intimacy is an essential aspect of romantic love, but it can also refer to the deep emotional closeness that develops over time with someone you care about deeply.
3. **Commitment**: Love often involves a willingness to commit to another person, making sacrifices for their well-being and happiness, and being willing to work through challenges together.
4. **Vulnerability**: Loving someone means being open and vulnerable, sharing your thoughts, feelings, and desires with them, and trusting that they will do the same in return.
5. **Unconditi

**🤔 생각해보기**: top_k=2일 때와 top_k=40일 때 결과가 어떻게 다른가요?  
top_k가 너무 작으면 어떤 문제가 생길 수 있을까요?

---
## 3. GPU(CUDA) 가속 사용

GPU가 있는 환경(Colab GPU 런타임, 또는 NVIDIA GPU가 탑재된 로컬 PC)에서는 CUDA 가속을 사용할 수 있습니다.

⚠️ **Colab 사용자**: 아래 셀을 실행하기 전에 **런타임 > 세션 다시 시작**을 해주세요!  
⚠️ **로컬 사용자**: 커널을 재시작한 후 아래부터 실행해주세요.

In [14]:
!pip install "gpt4all[cuda]"
# Apple Silicon(M1/M2/M3)을 사용하는 경우:
# !pip install "gpt4all[metal]"

In [15]:
from gpt4all import GPT4All
import time

model = GPT4All("Meta-Llama-3-8B-Instruct.Q4_0.gguf", device="cuda")
# Apple Silicon: device="metal"

RuntimeError: Unable to instantiate model: Could not find any implementations for backend: cuda

CPU로 돌렸을 때와 동일한 질문을 GPU로 돌려봅시다.  
시간을 측정해서 속도 차이를 확인합니다.

In [ ]:
start = time.time()
with model.chat_session():
    result = model.generate("How can I run LLMs efficiently on my laptop?", max_tokens=1024)
elapsed = time.time() - start

print(result)
print(f"\n⏱️ 소요 시간: {elapsed:.1f}초")

CPU에서는 약 1~2분 걸렸던 것이, GPU에서는 약 20\~30초로 단축됩니다.  
슬라이드에서 배운 GPU/CUDA의 효과를 직접 체감할 수 있습니다!

---
## 4. 자유 실습

아래 셀을 활용해서 다양한 실험을 해보세요!

**실습 과제:**
1. 질문을 바꿔서 모델의 응답을 확인해보세요 (한국어도 시도해보세요)
2. `temp`, `top_p`, `top_k` 를 동시에 조합해서 바꿔보세요
3. `max_tokens` 를 줄이거나 늘려보며 출력 길이를 조절해보세요
4. 모델의 응답에서 어떤 점이 부족한지 생각해보세요

In [ ]:
# 하이퍼파라미터 비교 실험
# 같은 질문으로 서로 다른 설정의 결과를 나란히 비교해봅시다

question = "What is the meaning of life?"

configs = [
    {"temp": 0.0, "top_k": 40,  "label": "보수적 (temp=0.0)"},
    {"temp": 0.7, "top_k": 40,  "label": "균형 (temp=0.7)"},
    {"temp": 1.0, "top_k": 40,  "label": "창의적 (temp=1.0)"},
    {"temp": 0.7, "top_k": 2,   "label": "제한적 (top_k=2)"},
]

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"설정: {cfg['label']}")
    print(f"{'='*60}")
    with model.chat_session():
        result = model.generate(
            question,
            max_tokens=150,
            temp=cfg["temp"],
            top_k=cfg["top_k"]
        )
        print(result)

---
## 정리

오늘 실습에서 확인한 것:

- `GPT4All` 라이브러리로 Python에서 로컬 LLM을 쉽게 사용할 수 있다
- `temp` → 높을수록 다양한 응답, 낮을수록 일관된 응답
- `top_p`, `top_k` → 낮을수록 보수적인 단어 선택
- `device="cuda"` → GPU 가속으로 5배 이상 속도 향상

**다음 시간에는:**  
GPT4ALL에서 사용 가능한 다양한 모델(LLaMA, Mistral, Phi)의 특성을 비교하고,  
모델마다 다른 **프롬프트 형식**을 이해합니다.